The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [1]:
%pip install nbformat plotly
%pip install umap-learn
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 29.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 41.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [umap-learn]4 [pynndescent]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [2]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
# TODO(you): build `vocab` (the V most common words), `word2idx`, `idx2word`,
# and `corpus` (the token stream mapped to ids, dropping out-of-vocabulary words).
vocab = [w for w, _ in counts.most_common(V)]
word2idx = dict((w, i) for i, w in enumerate(vocab))
idx2word = dict((i, w) for i, w in enumerate(vocab))
corpus = [word2idx[t] for t in tokens if t in word2idx]

/Users/johnromanazzi/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [3]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        # TODO(you): embed the center ids and return the (B, V) scores over the whole
        # vocabulary (one score per possible context word). Cross-entropy + softmax are applied
        # by the loss in the training loop, so return the raw scores (logits), not probabilities.
        z = self.center(center_ids)
        return self.output(z)

In [4]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

# TODO(you): write the training loop. For each mini-batch, get the (B, V) logits from the center
# ids with model(...), compute the cross-entropy loss against the true context ids with loss_fn,
# backprop, and step the optimizer. Track the per-epoch loss. After training, set
# `emb = model.center.weight.detach().cpu().numpy()`.
for epoch in range(epochs):
    np.random.shuffle(pairs)
    total = 0.0
    for k in range(0, len(pairs), B):
        batch = pairs[k:k + B]
        centers = torch.from_numpy(batch[:, 0])
        contexts = torch.from_numpy(batch[:, 1])
        logits = model(centers)
        loss = loss_fn(logits, contexts)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    print("EPOCH:", epoch, "LOSS:", total)

emb = model.center.weight.detach().cpu().numpy()

EPOCH: 0 LOSS: 11303.198276042938
EPOCH: 1 LOSS: 10861.191485881805
EPOCH: 2 LOSS: 10638.549812793732


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [5]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

# TODO(you): compute `pca3` (N x 3) with PCA, and `umap3` with UMAP (guard UMAP in a
# try/except so a missing umap-learn does not crash the notebook).
pca3 = PCA(n_components=3).fit_transform(X)

try:
    import umap
    umap3 = umap.UMAP(n_components=3).fit_transform(X)
except Exception:
    umap3 = None

In [6]:
import plotly.graph_objects as go

# TODO(you): write `plot_embeddings(coords, words, query=None, neighbor_set=None)` that
# draws a plotly Scatter3d: hover text = the word; color/size the `query` and any words in
# `neighbor_set` distinctly. Return the figure (end the cell with the figure object).
def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = neighbor_set or set()
    colors, sizes = [], []
    for w in words:
        if w == query:
            colors.append("crimson"); sizes.append(8)
        elif w in neighbor_set:
            colors.append("orange"); sizes.append(6)
        else:
            colors.append("steelblue"); sizes.append(3)
    fig = go.Figure(go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers", text=words, hoverinfo="text",
        marker=dict(color=colors, size=sizes),
    ))
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
    return fig

plot_embeddings(pca3, plot_words)

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [7]:
def neighbors(word, k=10):
    # TODO(you): return the k nearest words to `word` by cosine similarity over `emb`,
    # as a list of (word, score) sorted by descending score, excluding `word` itself.
    v_w1 = emb[word2idx[word]]
    word_sim = {}
    for i in range(V):
        v_w2 = emb[i]
        theta_num = np.dot(v_w1, v_w2)
        theta_den = np.linalg.norm(v_w1) * np.linalg.norm(v_w2)
        theta = theta_num / theta_den
        word_sim[idx2word[i]] = theta
    words_sorted = sorted(word_sim.items(), key=lambda item: item[1], reverse=True)
    return [(w, s) for w, s in words_sorted if w != word][:k]

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

federal         0.815
troops          0.814
municipal       0.797
authorities     0.793
pakistani       0.792
extending       0.789
subcontinent    0.788
courts          0.787
commonwealth    0.787
revolutionary   0.786


In [8]:
# TODO(you): pick a query word, get its neighbors with neighbors(query, 10), and re-draw the
# projector with plot_embeddings(...) highlighting the query and its neighbors. A word only
# appears in the plot if it is among the N most frequent words used for pca3.
query = "government"
neighbor_set = {w for w, _ in neighbors(query, 10)}
plot_embeddings(pca3, plot_words, query=query, neighbor_set=neighbor_set)

## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

In [9]:
for q in ["good", "river", "language", "played", "between", "house"]:
    if q not in word2idx:
        print(f"\n{q}  (not in vocab)")
        continue
    print(f"\n{q}")
    for w, s in neighbors(q, 8):
        print(f"  {w:15s} {s:.3f}")


good
  faithful        0.873
  extraordinary   0.855
  evil            0.842
  true            0.840
  overcome        0.836
  desire          0.834
  asking          0.822
  reading         0.820

river
  susquehanna     0.888
  confluence      0.834
  fort            0.812
  branch          0.810
  canal           0.800
  northwest       0.788
  crosses         0.787
  valley          0.778

language
  intelligentsia  0.774
  religious       0.770
  languages       0.768
  culture         0.763
  prewar          0.762
  protection      0.760
  censorship      0.759
  primary         0.756

played
  playing         0.795
  baseball        0.791
  starter         0.763
  junior          0.747
  pitched         0.745
  birthday        0.742
  fifa            0.733
  professional    0.731

between
  lies            0.690
  bonding         0.678
  vertically      0.672
  underparts      0.668
  separating      0.666
  spines          0.665
  crossed         0.663
  brawl           0.661


1. The more frequently used and specific words, like river or played, gave clean semantic neighborhoods. The context with these words are consistent so the embeddings are well defined. The broader words, like good, have looser neighborhoods because their context is more varied. Between was the noisiest as it is used in many contexts. A rare word would be noisier because it only appears in a few windows, getiing fewer gradient updates and staying more random. The neighbors do not reflect meaning as much as just sampling noise.

In [11]:
plot_embeddings(umap3, plot_words)

In [12]:
for q in ["river", "played", "king", "music", "german"]:
    print(f"\n{q}: " + ", ".join(w for w, _ in neighbors(q, 8)))


river: susquehanna, confluence, fort, branch, canal, northwest, crosses, valley

played: playing, baseball, starter, junior, pitched, birthday, fifa, professional

king: raymond, dragon, queensberry, herg, peter, grandson, tsar, nation

music: accompanying, concept, pop, susan, video, videos, best, sony

german: soviet, occupation, nazi, troops, occupying, invasion, russian, propaganda


2. Related words do land near each other. These groups appear as denser regions within one broad cloud rather than fully separated. The identifiable clusters could be geography with river (canal, valley, susquehanna), sports with played (baseball, pitched, fifa), and entertainment with music (pop, video, sony).